# Final Model Comparison: Drone and UAV Signal Classification

**Module 24 Technical Analysis**

This notebook completes the model-selection requirements for the capstone by comparing and tuning two classification models with five-fold cross-validation.

It uses the public [Micro-Doppler Aerial Classification Dataset](https://www.kaggle.com/datasets/mithula05/micro-doppler-aerial-classification-dataset) as a **secondary methodological benchmark**. This dataset is synthetic radar data, not the real RF communication data used in the primary capstone analysis. Its results are therefore reported separately and are not evidence that the primary RF detector will achieve the same performance.

**Secondary question:** Can engineered micro-Doppler summary features distinguish drone/UAV signatures from bird/aircraft signatures?


## 1. Setup

The compact feature file was generated from 2,800 rows of synthetic time-series data. Each original row contained 100 time steps for amplitude, velocity, and energy.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_PATH = PROJECT_ROOT / "reports" / "artifacts" / "micro_doppler" / "astra_summary_features.csv"
RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")

if not FEATURE_PATH.exists():
    raise FileNotFoundError(f"Missing feature file: {FEATURE_PATH}")

features = pd.read_csv(FEATURE_PATH)
print(f"Rows: {len(features):,}")
print(f"Columns: {features.shape[1]}")


## 2. Data structure and cleaning

The original labels are:

- 0: Bird
- 1: Drone
- 2: Aircraft
- 3: Stealth UAV

For binary classification, Drone and Stealth UAV are mapped to 1; Bird and Aircraft are mapped to 0. Before modeling, the analysis checks for missing values, duplicate rows, invalid labels, and nonnumeric feature values.


In [ ]:
label_names = {0: "Bird", 1: "Drone", 2: "Aircraft", 3: "Stealth UAV"}
features["class_name"] = features["original_label"].map(label_names)

cleaning_summary = pd.DataFrame({
    "check": ["Missing values", "Duplicate rows", "Invalid original labels"],
    "count": [
        int(features.isna().sum().sum()),
        int(features.duplicated().sum()),
        int((~features["original_label"].isin(label_names)).sum()),
    ],
})
display(cleaning_summary)

# Clean a copy so the source artifact remains unchanged.
model_data = features.drop_duplicates().dropna().copy()
model_data = model_data[model_data["original_label"].isin(label_names)]
print(f"Rows retained after cleaning: {len(model_data):,}")


## 3. Exploratory data analysis

The benchmark is balanced: each original class has 700 rows. This is useful for model comparison but does not represent real-world prevalence.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
order = ["Bird", "Drone", "Aircraft", "Stealth UAV"]
sns.countplot(data=model_data, x="class_name", order=order, color="#4C78A8", ax=axes[0])
axes[0].set(title="Samples by original class", xlabel="Class", ylabel="Samples")
axes[0].tick_params(axis="x", rotation=20)

binary_names = model_data["uav_associated"].map({0: "Bird / aircraft", 1: "Drone / UAV"})
sns.countplot(x=binary_names, color="#F58518", ax=axes[1])
axes[1].set(title="Binary target balance", xlabel="Target group", ylabel="Samples")
plt.tight_layout()
plt.show()


### Continuous-feature distributions

The boxplots compare engineered signal summaries across the four classes. Visible separation suggests that the synthetic generator created strong class patterns.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.boxplot(data=model_data, x="class_name", y="amplitude_std", order=order, ax=axes[0])
axes[0].set(title="Amplitude variation by class", xlabel="Class", ylabel="Amplitude standard deviation")
axes[0].tick_params(axis="x", rotation=20)

sns.boxplot(data=model_data, x="class_name", y="energy_mean", order=order, ax=axes[1])
axes[1].set(title="Mean energy by class", xlabel="Class", ylabel="Mean energy")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


### Correlations

The heatmap shows relationships among selected engineered features. Strong correlations are expected because several summaries come from the same underlying signal channel.


In [ ]:
selected_features = [
    "amplitude_mean", "amplitude_std", "amplitude_rms",
    "velocity_mean", "velocity_std", "velocity_rms",
    "energy_mean", "energy_std", "energy_rms",
]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(model_data[selected_features].corr(), cmap="vlag", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Correlation among selected engineered features")
plt.tight_layout()
plt.show()


## 4. Feature engineering

The raw benchmark contains 100 time steps for each of three channels: amplitude, velocity, and energy. Eight summaries were calculated for each channel:

- Mean
- Standard deviation
- Minimum
- Maximum
- Median
- 25th percentile
- 75th percentile
- Root mean square (RMS)

This produces 24 model features per observation. The binary target groups Drone and Stealth UAV together as UAV-associated activity.


In [ ]:
feature_columns = [c for c in model_data.columns if c not in ["original_label", "uav_associated", "class_name"]]
X = model_data[feature_columns]
y = model_data["uav_associated"]

print(f"Model features: {len(feature_columns)}")
print("Target counts:")
display(y.value_counts().rename(index={0: "Bird / aircraft", 1: "Drone / UAV"}).to_frame("rows"))


## 5. Train/test split and evaluation strategy

Twenty percent of the rows are held out for final testing. The remaining 80% are used in five-fold stratified cross-validation.

**Primary metric: recall.** Recall answers: “Of all drone/UAV examples, what proportion did the model identify?” Missing a UAV is the most important error for an awareness system. Precision, F1, PR-AUC, and ROC-AUC are also reported to prevent a one-metric interpretation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")


## 6. Multiple models and grid search

Two models are tuned:

- **Logistic regression:** a scaled linear baseline; grid search tests regularization strength and class weighting.
- **Random forest:** a nonlinear tree ensemble; grid search tests tree count, depth, leaf size, and class weighting.

GridSearchCV fits every parameter combination across five validation folds and selects the settings with the highest mean recall.


In [ ]:
searches = {
    "logistic_regression": GridSearchCV(
        Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]),
        {
            "classifier__C": [0.1, 1.0, 10.0],
            "classifier__class_weight": [None, "balanced"],
        },
        scoring="recall",
        cv=cv,
        n_jobs=1,
    ),
    "random_forest": GridSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
        {
            "n_estimators": [150, 300],
            "max_depth": [None, 10],
            "min_samples_leaf": [1, 2],
            "class_weight": [None, "balanced"],
        },
        scoring="recall",
        cv=cv,
        n_jobs=1,
    ),
}

results = []
predictions_by_model = {}
for model_name, search in searches.items():
    search.fit(X_train, y_train)
    probabilities = search.best_estimator_.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    predictions_by_model[model_name] = predictions
    results.append({
        "model": model_name,
        "best_cv_recall": search.best_score_,
        "test_recall": recall_score(y_test, predictions),
        "test_precision": precision_score(y_test, predictions),
        "test_f1": f1_score(y_test, predictions),
        "test_pr_auc": average_precision_score(y_test, probabilities),
        "test_roc_auc": roc_auc_score(y_test, probabilities),
        "best_parameters": search.best_params_,
    })

comparison = pd.DataFrame(results).sort_values(["best_cv_recall", "test_pr_auc"], ascending=False)
display(comparison)


## 7. Model comparison

Cross-validation recall and held-out recall are compared below. Both models perform almost perfectly, so the simpler logistic regression is preferred under a parsimony rule. More complexity is not justified when it does not materially improve validation performance.


In [ ]:
plot_data = comparison.melt(
    id_vars="model",
    value_vars=["best_cv_recall", "test_recall"],
    var_name="evaluation",
    value_name="recall",
)
fig, ax = plt.subplots(figsize=(8, 4.8))
sns.barplot(data=plot_data, x="model", y="recall", hue="evaluation", ax=ax)
ax.set(title="Cross-validation and held-out recall by model", xlabel="Model", ylabel="Recall", ylim=(0, 1.05))
ax.legend(title="Evaluation", labels=["5-fold CV", "Held-out test"])
plt.tight_layout()
plt.show()


## 8. Selected model evaluation

Logistic regression is selected because it matches random forest performance while being easier to explain. The confusion matrix shows one false positive and no missed drone/UAV examples in the held-out synthetic test set.


In [ ]:
selected_model = "logistic_regression"
selected_predictions = predictions_by_model[selected_model]

fig, ax = plt.subplots(figsize=(5.8, 4.8))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    selected_predictions,
    display_labels=["Bird / aircraft", "Drone / UAV"],
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("Held-out confusion matrix: Logistic regression")
plt.tight_layout()
plt.show()

print(classification_report(
    y_test,
    selected_predictions,
    target_names=["Bird / aircraft", "Drone / UAV"],
    digits=3,
))


## 9. Findings and limitations

**Technical finding:** On this synthetic secondary benchmark, both tuned models achieved 100% five-fold cross-validation recall and 100% held-out recall. Logistic regression produced one false positive and no false negatives.

**Interpretation:** These unusually high scores mainly show that the synthetic classes have strongly separated signal patterns. They do not demonstrate real-world deployment readiness.

**Relationship to the primary RF analysis:**

- The real Noisy Drone RF baseline remains the primary capstone evidence.
- That real-data baseline achieved 64.5% recall and a 3.5% false-positive rate.
- The synthetic benchmark demonstrates model comparison, cross-validation, and tuning methodology.
- Results from radar micro-Doppler data cannot be transferred directly to RF communication-signal detection.

**Recommendation:** Preserve the real RF result as the operational baseline. Future work should reacquire the real RF feature matrix and repeat the same five-fold grid-search workflow on that source before selecting a production candidate.


## 10. Model considered but not selected

YOLO was considered as a possible advanced model but was not selected. YOLO is primarily an object-detection method that locates objects inside images using bounding boxes. This project assigns one class to an entire RF or micro-Doppler signal and has no bounding-box annotations. Therefore, YOLO does not match the structure of the data or the research question. If deep learning is explored later, a CNN that classifies complete spectrogram images would be more appropriate.
